# DroneDetect V2 - CNN Classification

Train CNN classifiers on spectrogram features:
- VGG16 (frozen features + trainable FC)
- ResNet50 (frozen features + trainable FC)
- File-level stratified split to prevent data leakage
- Early stopping + ReduceLROnPlateau(val_loss)
- Side-by-side performance comparison

## 0. Colab Setup

In [1]:
import os
import subprocess

COLAB = "COLAB_RELEASE_TAG" in os.environ

if COLAB:
    subprocess.check_call(["pip", "install", "-q", "boto3", "kaleido"])
    from google.colab import userdata
    import boto3

    s3 = boto3.client(
        "s3",
        endpoint_url="https://s3.fr-par.scw.cloud",
        region_name="fr-par",
        aws_access_key_id=userdata.get("SCW_ACCESS_KEY"),
        aws_secret_access_key=userdata.get("SCW_SECRET_KEY"),
    )
    ARTIFACT_BUCKET = "mldrone-artefacts"

    os.makedirs("../data/features", exist_ok=True)
    os.makedirs("../data/models", exist_ok=True)

    downloads = {
        "features/spectrogram_features.npy": "../data/features/spectrogram_features.npy",
        "features/spectrogram_meta.npz": "../data/features/spectrogram_meta.npz",
        "split/split_indices.npz": "../data/split_indices.npz",
    }
    for s3_key, local_path in downloads.items():
        if not os.path.exists(local_path):
            print(f"Downloading {s3_key}...")
            s3.download_file(ARTIFACT_BUCKET, s3_key, local_path)
            print(f"Done: {local_path}")

Done: ../data/features/spectrogram_features.npy
Done: ../data/features/spectrogram_meta.npz
Done: ../data/split_indices.npz


## 1. Imports and Configuration

In [2]:
import copy
import gc
import logging
import re
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, f1_score
from tqdm import tqdm
import torchvision.models as tv_models

try:
    from dronedetect.splitting import create_stratified_split, load_split, verify_split_balance
except ImportError:
    from sklearn.model_selection import StratifiedGroupKFold

    def create_stratified_split(
        y: np.ndarray,
        file_ids: np.ndarray,
        train_size: float = 0.70,
        val_size: float = 0.15,
        test_size: float = 0.15,
        random_state: int = 42,
        save_path: str | Path | None = None,
    ) -> dict:
        """
        Two-stage stratified group split into train/val/test partitions.

        Stage 1: split off test set using StratifiedGroupKFold.
        Stage 2: split remaining trainval into train and val.

        Stratification is by drone label (y). Grouping is by file_ids to prevent
        segments from the same recording appearing in different partitions.

        Args:
            y: Drone labels array, shape (n_samples,).
            file_ids: File identifier per sample, shape (n_samples,).
            train_size: Fraction of data for training.
            val_size: Fraction of data for validation.
            test_size: Fraction of data for testing.
            random_state: Seed for reproducibility.
            save_path: If provided, save split indices to this .npz path.

        Returns:
            Dict with keys: train_idx, val_idx, test_idx, split_metadata.

        Raises:
            ValueError: If size fractions do not sum to 1 or leakage is detected.
        """
        y = np.asarray(y)
        file_ids = np.asarray(file_ids)

        total = train_size + val_size + test_size
        if not np.isclose(total, 1.0):
            raise ValueError(f"Split fractions must sum to 1.0, got {total:.4f}")

        n_splits_test = round(1 / test_size)
        sgkf_test = StratifiedGroupKFold(
            n_splits=n_splits_test, shuffle=True, random_state=random_state
        )
        all_indices = np.arange(len(y))

        trainval_idx, test_idx = next(sgkf_test.split(all_indices, y, groups=file_ids))

        relative_val = val_size / (train_size + val_size)
        n_splits_val = round(1 / relative_val)
        sgkf_val = StratifiedGroupKFold(
            n_splits=n_splits_val, shuffle=True, random_state=random_state + 1
        )

        y_trainval = y[trainval_idx]
        file_ids_trainval = file_ids[trainval_idx]

        train_local, val_local = next(
            sgkf_val.split(
                np.arange(len(trainval_idx)), y_trainval, groups=file_ids_trainval
            )
        )
        train_idx = trainval_idx[train_local]
        val_idx = trainval_idx[val_local]

        train_files = set(file_ids[train_idx])
        val_files = set(file_ids[val_idx])
        test_files = set(file_ids[test_idx])

        overlap_tv = train_files & val_files
        overlap_tt = train_files & test_files
        overlap_vt = val_files & test_files

        if overlap_tv or overlap_tt or overlap_vt:
            msg = "Data leakage detected! File overlap between partitions:"
            if overlap_tv:
                msg += f"\n  train-val: {overlap_tv}"
            if overlap_tt:
                msg += f"\n  train-test: {overlap_tt}"
            if overlap_vt:
                msg += f"\n  val-test: {overlap_vt}"
            raise ValueError(msg)

        logger.info("Split created with zero file overlap (leakage-free)")
        logger.info(
            "Samples  -> train=%d, val=%d, test=%d (total=%d)",
            len(train_idx),
            len(val_idx),
            len(test_idx),
            len(y),
        )
        logger.info(
            "Files    -> train=%d, val=%d, test=%d",
            len(train_files),
            len(val_files),
            len(test_files),
        )

        split_metadata = {
            "train_samples": len(train_idx),
            "val_samples": len(val_idx),
            "test_samples": len(test_idx),
            "train_files": len(train_files),
            "val_files": len(val_files),
            "test_files": len(test_files),
            "random_state": random_state,
        }

        result = {
            "train_idx": train_idx,
            "val_idx": val_idx,
            "test_idx": test_idx,
            "split_metadata": split_metadata,
        }

        if save_path is not None:
            save_path = Path(save_path)
            save_path.parent.mkdir(parents=True, exist_ok=True)
            np.savez(
                save_path,
                train_idx=train_idx,
                val_idx=val_idx,
                test_idx=test_idx,
            )
            logger.info("Split indices saved to %s", save_path)

        return result

    def verify_split_balance(
        file_ids: np.ndarray,
        y: np.ndarray,
        split_indices: dict,
        conditions: np.ndarray | None = None,
    ) -> None:
        """
        Log class and condition distributions per partition for diagnostics.

        Args:
            file_ids: File identifier per sample, shape (n_samples,).
            y: Drone labels array, shape (n_samples,).
            split_indices: Dict with train_idx, val_idx, test_idx.
            conditions: Optional condition labels (e.g. CLEAN/WIFI/BLUE/BOTH).
        """
        y = np.asarray(y)
        file_ids = np.asarray(file_ids)
        if conditions is not None:
            conditions = np.asarray(conditions)

        partitions = {
            "train": split_indices["train_idx"],
            "val": split_indices["val_idx"],
            "test": split_indices["test_idx"],
        }

        all_classes = np.unique(y)

        for name, idx in partitions.items():
            y_part = y[idx]
            classes, counts = np.unique(y_part, return_counts=True)
            dist_str = ", ".join(f"{c}={n}" for c, n in zip(classes, counts))
            logger.info("[%s] Class distribution: %s", name, dist_str)

        if conditions is not None:
            all_conditions = np.unique(conditions)

            for name, idx in partitions.items():
                cond_part = conditions[idx]
                conds, counts = np.unique(cond_part, return_counts=True)
                dist_str = ", ".join(f"{c}={n}" for c, n in zip(conds, counts))
                logger.info("[%s] Condition distribution: %s", name, dist_str)

                y_part = y[idx]
                for drone in all_classes:
                    drone_mask = y_part == drone
                    drone_conditions = set(np.unique(cond_part[drone_mask]))
                    missing = set(all_conditions) - drone_conditions
                    if missing:
                        logger.warning(
                            "[%s] Drone '%s' missing conditions: %s",
                            name,
                            drone,
                            missing,
                        )

    def load_split(path: str | Path) -> dict:
        """
        Load previously saved split indices from a .npz file.

        Args:
            path: Path to the .npz file created by create_stratified_split.

        Returns:
            Dict with keys: train_idx, val_idx, test_idx.
        """
        data = np.load(path)
        return {
            "train_idx": data["train_idx"],
            "val_idx": data["val_idx"],
            "test_idx": data["test_idx"],
        }

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logger.info("Using device: %s", device)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

NOTEBOOK_NAME = "training_cnn"
FIGURES_DIR = Path("figures") / NOTEBOOK_NAME


def save_figure(fig) -> None:
    """Save plotly figure to PNG file using the figure's title as filename."""
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    title = fig.layout.title.text if fig.layout.title.text else "untitled"
    filename = re.sub(r'[^\w\s-]', '', title).strip()
    filename = re.sub(r'[\s-]+', '_', filename)
    filepath = FIGURES_DIR / f"{filename}.png"
    try:
        fig.write_image(str(filepath), width=1200, height=800)
        logger.info("Saved: %s", filepath)
    except Exception as e:
        logger.warning("Could not save figure (kaleido required): %s", e)

In [3]:
FEATURES_DIR = Path("../data/features")
SPLIT_PATH = Path("../data/split_indices.npz")

CONFIG = {
    'models_dir': '../data/models/',
    'test_data_dir': '../data/sample/test_data/',

    'batch_size': 128,
    'max_epochs': 120,
    'patience': 5,
    # Transfer learning requires lower LR to preserve pretrained features
    'lr_vgg16': 1e-3,
    'lr_resnet50': 1e-4,

    'device': device,
}

logger.info("Configuration: %s", CONFIG)

## 2. Model Definitions

In [4]:
class VGG16FC(nn.Module):
    """VGG16 with frozen features and trainable classifier.

    Uses a pre-trained VGG16 backbone with weights frozen,
    replacing the classifier with a new fully connected layer.
    """

    def __init__(self, num_classes: int):
        super().__init__()
        vgg = tv_models.vgg16(weights='IMAGENET1K_V1')
        self.features = nn.Sequential(*list(vgg.children())[:-1])

        for param in self.features.parameters():
            param.requires_grad = False

        self.classifier = nn.Linear(25088, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.dim() == 4 and x.shape[-1] == 3:
            x = x.permute(0, 3, 1, 2)

        x = self.features(x)
        x = x.flatten(1)
        return self.classifier(x)


class ResNet50FC(nn.Module):
    """ResNet50 with frozen features and trainable classifier.

    Uses a pre-trained ResNet50 backbone with weights frozen.
    AdaptiveAvgPool2d reduces feature maps to (1,1) spatially,
    keeping FC at 2048 inputs instead of 100352 (49x param reduction).
    """

    def __init__(self, num_classes: int):
        super().__init__()
        resnet = tv_models.resnet50(weights='IMAGENET1K_V1')
        self.features = nn.Sequential(*list(resnet.children())[:-2])

        for param in self.features.parameters():
            param.requires_grad = False

        # Spatial pooling prevents FC explosion: 2048x7x7 -> 2048x1x1
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Linear(2048, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.dim() == 4 and x.shape[-1] == 3:
            x = x.permute(0, 3, 1, 2)

        x = self.features(x)
        x = self.pool(x)
        x = x.flatten(1)
        return self.classifier(x)

## 3. Memory-Efficient Dataset

**Problem**: Spectrogram .npy file is ~11 GB (19478 samples x 224x224x3 x 4 bytes).
Using fancy indexing `X_train = X[train_idx]` on a memory-mapped array forces NumPy
to load the entire subset into RAM, causing OOM.

**Solution**: Custom `RGBMemmapDataset` that stores indices and loads one sample
at a time via `__getitem__`. The `.copy()` call breaks the memmap view and loads
only the requested sample.

**Trade-off**: Disk I/O overhead (~10-20s per epoch), but prevents OOM crashes.

In [5]:
class RGBMemmapDataset(Dataset):
    """Memory-efficient dataset for memmap RGB spectrograms.

    Loads one sample at a time from disk instead of the entire array into RAM.
    Stores indices and loads samples individually via __getitem__, avoiding
    fancy indexing on memmap which would materialize the full subset.

    Parameters
    ----------
    memmap_array : np.memmap
        Memory-mapped array from np.load(..., mmap_mode='r')
    indices : np.ndarray
        Indices for this split (train, val, or test)
    labels : np.ndarray
        Labels for samples at those indices
    """

    def __init__(self, memmap_array, indices, labels):
        self.memmap = memmap_array
        self.indices = indices
        self.labels = labels

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        actual_idx = self.indices[idx]
        # .copy() breaks memmap view to avoid loading entire array into RAM
        rgb = self.memmap[actual_idx].copy()

        x = torch.from_numpy(rgb).float()
        y_label = torch.tensor(self.labels[idx]).long()

        return x, y_label

## 4. Load Spectrogram Features

In [6]:
# mmap_mode avoids loading full ~11 GB array into RAM
X_memmap = np.load(FEATURES_DIR / "spectrogram_features.npy", mmap_mode="r")
meta = np.load(FEATURES_DIR / "spectrogram_meta.npz")

y_drone = meta["y_drone"]
y_interference = meta["y_interference"]
y_state = meta["y_state"]
file_ids = meta["file_ids"]
# Copy string arrays from npz into RAM to avoid repeated disk access
drone_classes = np.array(meta["drone_classes"])
interference_classes = np.array(meta["interference_classes"])
state_classes = np.array(meta["state_classes"])

logger.info("Spectrograms shape: %s", X_memmap.shape)
logger.info("Labels shape: %s", y_drone.shape)
logger.info("File IDs: %d unique files", len(np.unique(file_ids)))
logger.info("Drone classes: %s", drone_classes)
logger.info("Number of classes: %d", len(drone_classes))

In [7]:
if SPLIT_PATH.exists():
    split = load_split(SPLIT_PATH)
    logger.info("Loaded existing split from %s", SPLIT_PATH)
else:
    split = create_stratified_split(y_drone, file_ids, save_path=SPLIT_PATH)
    logger.info("Created and saved new split to %s", SPLIT_PATH)

train_idx = split["train_idx"]
val_idx = split["val_idx"]
test_idx = split["test_idx"]

verify_split_balance(file_ids, y_drone, split, conditions=y_interference)

y_train = y_drone[train_idx]
y_val = y_drone[val_idx]
y_test = y_drone[test_idx]

logger.info("Train: %d | Val: %d | Test: %d samples", len(train_idx), len(val_idx), len(test_idx))

# Save bulk test data for post-training evaluation
test_data_dir = Path(CONFIG['test_data_dir'])
test_data_dir.mkdir(parents=True, exist_ok=True)

X_test = X_memmap[test_idx].copy()
test_data_path = test_data_dir / "cnn_test_data.npz"
np.savez(
    test_data_path,
    X_test=X_test,
    y_test=y_test,
    y_interference_test=y_interference[test_idx],
    y_state_test=y_state[test_idx],
    test_idx=test_idx,
    file_ids_test=file_ids[test_idx],
    drone_classes=drone_classes,
    interference_classes=interference_classes,
    state_classes=state_classes,
)
logger.info("Test data saved to %s", test_data_path)

In [8]:
train_dataset = RGBMemmapDataset(X_memmap, train_idx, y_train)
val_dataset = RGBMemmapDataset(X_memmap, val_idx, y_val)
test_dataset = RGBMemmapDataset(X_memmap, test_idx, y_test)

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

logger.info("Train batches: %d | Val batches: %d | Test batches: %d",
            len(train_loader), len(val_loader), len(test_loader))

## 5. Training Function

In [9]:
def train_model(model, train_loader, val_loader, max_epochs=30, patience=5,
                lr=1e-3, device='cuda'):
    """Train a PyTorch model with early stopping and LR scheduling on val loss.

    Parameters
    ----------
    model : nn.Module
        Model to train.
    train_loader : DataLoader
        Training data loader.
    val_loader : DataLoader
        Validation data loader (used for LR scheduling and early stopping).
    max_epochs : int
        Maximum number of training epochs.
    patience : int
        Early stopping patience (epochs without val_loss improvement).
    lr : float
        Initial learning rate.
    device : str
        Device to use ('cuda' or 'cpu').

    Returns
    -------
    model : nn.Module
        Best model (lowest val_loss), restored after training.
    history : dict
        Training history with train/val metrics and LR per epoch.
    """
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    # val_loss instead of test_loss to prevent information leakage
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=2
    )

    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': [],
        'lr': [],
    }

    best_val_loss = float('inf')
    epochs_without_improvement = 0
    best_model_state = None

    for epoch in range(max_epochs):
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for batch_X, batch_y in tqdm(train_loader, desc=f"Epoch {epoch+1}/{max_epochs}"):
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            train_total += batch_y.size(0)
            train_correct += (predicted == batch_y).sum().item()

        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                batch_X = batch_X.to(device)
                batch_y = batch_y.to(device)

                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)

                val_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                val_total += batch_y.size(0)
                val_correct += (predicted == batch_y).sum().item()

        avg_val_loss = val_loss / len(val_loader)
        scheduler.step(avg_val_loss)

        history['train_loss'].append(train_loss / len(train_loader))
        history['train_acc'].append(100 * train_correct / train_total)
        history['val_loss'].append(avg_val_loss)
        history['val_acc'].append(100 * val_correct / val_total)
        history['lr'].append(optimizer.param_groups[0]['lr'])

        logger.info(
            "Epoch %d/%d: Train Loss=%.4f, Train Acc=%.2f%%, "
            "Val Loss=%.4f, Val Acc=%.2f%%, LR=%.2e",
            epoch + 1, max_epochs,
            history['train_loss'][-1], history['train_acc'][-1],
            history['val_loss'][-1], history['val_acc'][-1],
            history['lr'][-1],
        )

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            epochs_without_improvement = 0
            # deepcopy required: state_dict() returns references, not copies
            best_model_state = copy.deepcopy(model.state_dict())
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            logger.info("Early stopping at epoch %d (patience=%d)", epoch + 1, patience)
            break

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Restore best model weights
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        logger.info("Restored best model (val_loss=%.4f)", best_val_loss)

    return model, history

## 6. Train VGG16

In [10]:
num_classes = len(drone_classes)
vgg_model = VGG16FC(num_classes=num_classes)

logger.info("Training VGG16 with %d classes...", num_classes)
vgg_model, vgg_history = train_model(
    vgg_model,
    train_loader,
    val_loader,
    max_epochs=CONFIG['max_epochs'],
    patience=CONFIG['patience'],
    lr=CONFIG['lr_vgg16'],
    device=CONFIG['device'],
)
logger.info("VGG16 training complete (%d epochs)", len(vgg_history['train_loss']))

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:02<00:00, 200MB/s]
Epoch 12/120: 100%|██████████| 218/218 [00:15<00:00, 13.69it/s]


## 7. VGG16 Performance Metrics

In [11]:
vgg_model.eval()
vgg_preds = []
vgg_true = []

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X = batch_X.to(CONFIG['device'])
        outputs = vgg_model(batch_X)
        _, predicted = torch.max(outputs, 1)
        vgg_preds.extend(predicted.cpu().numpy())
        vgg_true.extend(batch_y.numpy())

vgg_preds = np.array(vgg_preds)
vgg_true = np.array(vgg_true)

present_labels = sorted(set(vgg_true) | set(vgg_preds))
present_names = [drone_classes[i] for i in present_labels]

print("VGG16 Classification Report:")
print(classification_report(vgg_true, vgg_preds, labels=present_labels, target_names=present_names))
print(f"VGG16 Accuracy: {accuracy_score(vgg_true, vgg_preds):.4f}")
print(f"VGG16 F1 Score (macro): {f1_score(vgg_true, vgg_preds, average='macro'):.4f}")

VGG16 Classification Report:
              precision    recall  f1-score   support

         AIR       0.95      0.80      0.87       922
         DIS       0.94      0.94      0.94       600
         INS       0.92      0.96      0.94       800
         MA1       0.81      0.76      0.79      1000
         MAV       0.69      0.85      0.76       800
         MIN       1.00      1.00      1.00       800
         PHA       0.97      0.98      0.98       700

    accuracy                           0.89      5622
   macro avg       0.90      0.90      0.90      5622
weighted avg       0.89      0.89      0.89      5622

VGG16 Accuracy: 0.8876
VGG16 F1 Score (macro): 0.8960


In [12]:
cm_vgg = confusion_matrix(vgg_true, vgg_preds)

fig = go.Figure(data=go.Heatmap(
    z=cm_vgg,
    x=list(drone_classes),
    y=list(drone_classes),
    colorscale='Blues',
    text=cm_vgg,
    texttemplate='%{text}',
    textfont={'size': 12},
    hoverongaps=False
))

fig.update_layout(
    title='VGG16 Confusion Matrix',
    xaxis_title='Predicted Label',
    yaxis_title='True Label',
    xaxis={'side': 'bottom'},
    yaxis={'autorange': 'reversed'},
    width=800,
    height=700
)
fig.show()
save_figure(fig)

/usr/local/lib/python3.12/dist-packages/kaleido/_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.

You can however, use the Kaleido API directly which will work with your plotly version. `kaleido.write_fig(...)`, for example. Please see the kaleido documentation.




Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



## 8. Train ResNet50

In [13]:
num_classes = len(drone_classes)
resnet_model = ResNet50FC(num_classes=num_classes)

logger.info("Training ResNet50 with %d classes...", num_classes)
resnet_model, resnet_history = train_model(
    resnet_model,
    train_loader,
    val_loader,
    max_epochs=CONFIG['max_epochs'],
    patience=CONFIG['patience'],
    lr=CONFIG['lr_resnet50'],
    device=CONFIG['device'],
)
logger.info("ResNet50 training complete (%d epochs)", len(resnet_history['train_loss']))

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 241MB/s]
Epoch 26/120: 100%|██████████| 218/218 [00:11<00:00, 18.32it/s]


## 9. ResNet50 Performance Metrics

In [14]:
resnet_model.eval()
resnet_preds = []
resnet_true = []

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X = batch_X.to(CONFIG['device'])
        outputs = resnet_model(batch_X)
        _, predicted = torch.max(outputs, 1)
        resnet_preds.extend(predicted.cpu().numpy())
        resnet_true.extend(batch_y.numpy())

resnet_preds = np.array(resnet_preds)
resnet_true = np.array(resnet_true)

present_labels = sorted(set(resnet_true) | set(resnet_preds))
present_names = [drone_classes[i] for i in present_labels]

print("ResNet50 Classification Report:")
print(classification_report(resnet_true, resnet_preds, labels=present_labels, target_names=present_names))
print(f"ResNet50 Accuracy: {accuracy_score(resnet_true, resnet_preds):.4f}")
print(f"ResNet50 F1 Score (macro): {f1_score(resnet_true, resnet_preds, average='macro'):.4f}")

ResNet50 Classification Report:
              precision    recall  f1-score   support

         AIR       0.84      0.73      0.78       922
         DIS       0.94      0.82      0.88       600
         INS       0.69      0.83      0.75       800
         MA1       0.69      0.62      0.65      1000
         MAV       0.61      0.68      0.64       800
         MIN       0.97      0.99      0.98       800
         PHA       0.90      0.92      0.91       700

    accuracy                           0.79      5622
   macro avg       0.81      0.80      0.80      5622
weighted avg       0.80      0.79      0.79      5622

ResNet50 Accuracy: 0.7880
ResNet50 F1 Score (macro): 0.7990


## 10. ResNet50 Confusion Matrix

In [15]:
cm_resnet = confusion_matrix(resnet_true, resnet_preds)

fig = go.Figure(data=go.Heatmap(
    z=cm_resnet,
    x=list(drone_classes),
    y=list(drone_classes),
    colorscale='Greens',
    text=cm_resnet,
    texttemplate='%{text}',
    textfont={'size': 12},
    hoverongaps=False
))

fig.update_layout(
    title='ResNet50 Confusion Matrix',
    xaxis_title='Predicted Label',
    yaxis_title='True Label',
    xaxis={'side': 'bottom'},
    yaxis={'autorange': 'reversed'},
    width=800,
    height=700
)
fig.show()
save_figure(fig)

Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



## 11. Side-by-Side Performance Comparison

In [16]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Training History Comparison', 'Final Performance Comparison'),
)

epoch_range = list(range(1, len(vgg_history['train_acc']) + 1))

fig.add_trace(go.Scatter(x=epoch_range, y=vgg_history['train_acc'], mode='lines+markers',
                         name='VGG16 Train', line=dict(color='blue')), row=1, col=1)
fig.add_trace(go.Scatter(x=epoch_range, y=vgg_history['val_acc'], mode='lines+markers',
                         name='VGG16 Val', line=dict(color='blue', dash='dash')), row=1, col=1)

resnet_epoch_range = list(range(1, len(resnet_history['train_acc']) + 1))
fig.add_trace(go.Scatter(x=resnet_epoch_range, y=resnet_history['train_acc'], mode='lines+markers',
                         name='ResNet50 Train', line=dict(color='green')), row=1, col=1)
fig.add_trace(go.Scatter(x=resnet_epoch_range, y=resnet_history['val_acc'], mode='lines+markers',
                         name='ResNet50 Val', line=dict(color='green', dash='dash')), row=1, col=1)

models = ['VGG16', 'ResNet50']
accuracies = [accuracy_score(vgg_true, vgg_preds), accuracy_score(resnet_true, resnet_preds)]
f1_scores_vals = [f1_score(vgg_true, vgg_preds, average='macro'),
                  f1_score(resnet_true, resnet_preds, average='macro')]

fig.add_trace(go.Bar(x=models, y=accuracies, name='Accuracy', marker_color='steelblue'), row=1, col=2)
fig.add_trace(go.Bar(x=models, y=f1_scores_vals, name='F1 Score (macro)', marker_color='coral'), row=1, col=2)

fig.update_layout(
    title='CNN Training Comparison - VGG16 vs ResNet50',
    height=500,
    width=1200,
    barmode='group',
)
fig.update_xaxes(title_text='Epoch', row=1, col=1)
fig.update_yaxes(title_text='Accuracy (%)', row=1, col=1)
fig.update_xaxes(title_text='Model', row=1, col=2)
fig.update_yaxes(title_text='Score', row=1, col=2)

fig.show()
save_figure(fig)

print("\nFinal Results (on test set):")
print(f"VGG16    - Accuracy: {accuracies[0]:.4f}, F1: {f1_scores_vals[0]:.4f}")
print(f"ResNet50 - Accuracy: {accuracies[1]:.4f}, F1: {f1_scores_vals[1]:.4f}")

Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido




Final Results (on test set):
VGG16    - Accuracy: 0.8876, F1: 0.8960
ResNet50 - Accuracy: 0.7880, F1: 0.7990


## 12. Save Models

In [17]:
models_dir = Path(CONFIG['models_dir'])
models_dir.mkdir(parents=True, exist_ok=True)

vgg_path = models_dir / 'vgg16_cnn.pth'
torch.save({
    'model_state_dict': vgg_model.state_dict(),
    'history': vgg_history,
    'num_classes': num_classes,
    'drone_classes': drone_classes,
}, vgg_path)
logger.info("VGG16 model saved to %s", vgg_path)

resnet_path = models_dir / 'resnet50_cnn.pth'
torch.save({
    'model_state_dict': resnet_model.state_dict(),
    'history': resnet_history,
    'num_classes': num_classes,
    'drone_classes': drone_classes,
}, resnet_path)
logger.info("ResNet50 model saved to %s", resnet_path)

## 13. Upload Models to Scaleway

In [18]:
if COLAB:
    import glob
    model_dir = CONFIG["models_dir"] if "CONFIG" in dir() else "../data/models"
    for model_file in sorted(glob.glob(os.path.join(model_dir, "*.pth"))):
        model_name = os.path.basename(model_file)
        s3.upload_file(model_file, ARTIFACT_BUCKET, f"models/{model_name}")
        print(f"Uploaded {model_name}")

Uploaded resnet50_cnn.pth
Uploaded vgg16_cnn.pth


## 14. Summary

In [19]:
print("=" * 60)
print("CNN TRAINING SUMMARY")
print("=" * 60)

print(f"\nDataset: {len(y_drone)} total | {len(y_train)} train | {len(y_val)} val | {len(y_test)} test | {num_classes} classes")
print(f"Training: max_epochs={CONFIG['max_epochs']}, patience={CONFIG['patience']}, batch_size={CONFIG['batch_size']}")
print(f"  VGG16 LR: {CONFIG['lr_vgg16']}, ResNet50 LR: {CONFIG['lr_resnet50']}")
print(f"  VGG16 stopped at epoch {len(vgg_history['train_loss'])}")
print(f"  ResNet50 stopped at epoch {len(resnet_history['train_loss'])}")

print("\nTest set metrics:")
print(f"  VGG16:    Accuracy={accuracy_score(vgg_true, vgg_preds):.4f}, F1={f1_score(vgg_true, vgg_preds, average='macro'):.4f}")
print(f"  ResNet50: Accuracy={accuracy_score(resnet_true, resnet_preds):.4f}, F1={f1_score(resnet_true, resnet_preds, average='macro'):.4f}")

print(f"\nModels saved to: {CONFIG['models_dir']}")
print("=" * 60)

CNN TRAINING SUMMARY

Dataset: 39000 total | 27778 train | 5600 val | 5622 test | 7 classes
Training: max_epochs=120, patience=5, batch_size=128
  VGG16 LR: 0.001, ResNet50 LR: 0.0001
  VGG16 stopped at epoch 12
  ResNet50 stopped at epoch 26

Test set metrics:
  VGG16:    Accuracy=0.8876, F1=0.8960
  ResNet50: Accuracy=0.7880, F1=0.7990

Models saved to: ../data/models/
